In [0]:
%run "./environmental_variables"

In [0]:
'''what: Clean data, fix types, remove nulls, CDC ,standardize columns
   how: Reading the bronze delta files 
   why: To Transform the raw data into into transactional level data
'''

In [0]:
'''For dimentional or master data like customers, franchise, and suppliers, I implemented CDC using Delta MERGE with SCD Type 1. This ensures the Silver layer always contains the latest, cleanest version of the data while keeping Bronze as the raw immutable source.'''

from pyspark.sql.functions import *

from delta.tables import DeltaTable 
import os 
def read_bronze_CDC(table_name):
  silver_path = f"{config['silver_path']}{table_name}"
  silver_table=f"{config['database']}{table_name}"
  bronze_path = f"{config['bronze_path']}{table_name}"
  if table_name == 'sales_customers':
    id = 'customerid'
  elif table_name == 'sales_franchise':
    id = 'franchiseid'
  elif table_name == 'sales_suppliers':
    id = 'supplierid'

  if not spark.catalog.tableExists(silver_table):
    print(f"first time load for {table_name}")
    source_val= spark.read.format("delta").load(bronze_path)
    source_val.write.format("delta").saveAsTable(silver_table)
  else:
    print(f" CDC load for {table_name}")
    # # update_cols = {c: f "s.{c}" for c in source_val.columns if c != id}
    delta_table = DeltaTable.forName(spark, silver_table)
    source_val= spark.read.format("delta").load(bronze_path)
    def build_update_mapping(source_df, id_col):
        # Exclude the ID column
        update_cols = [c for c in source_df.columns if c != id_col]
        # Build mapping: target_col -> source_col
        update_dict = {c: f"s.{c}" for c in update_cols}
        # Build condition: update only if at least one source column is non-null
        non_null_checks = [f"s.{c} IS NOT NULL" for c in update_cols]
        condition = " OR ".join(non_null_checks)
        return update_dict, condition
      
      
    update_dict, condition = build_update_mapping(source_val, id)
    (
    delta_table.alias("t")
    .merge(source_val.alias("s"), f"t.{id} = s.{id}")
    .whenMatchedUpdate(condition=condition, set=update_dict)
    .whenNotMatchedInsertAll()
    .execute()
    )



for table in cdc:
    read_bronze_CDC(table)


In [0]:
'''Applying masking statergy for cardnumber in silver layer to handle the PII data and droping the original cardnumber  
'''

from pyspark.sql.functions import *
transactions_bronzepath= f"{config['bronze_path']}sales_transactions"
sales_transactions = spark.read.format('delta').load(transactions_bronzepath)\
    .withColumn('masked_cardnumber', concat(substring('cardnumber', 0, 4),lit('-XXXX-XXXX-XXXX'))).drop('cardnumber')

# display(sales_transactions)

In [0]:
'''Renaming the same column names between Franchise and Suppliers to avoid conflict while joining
'''

franchise_silver= f"{config['database']}sales_franchise"
# franchise_silver= f"inceptezcatalog.bakehouse.sales_franchise"


franchise_silver= spark.read.table(franchise_silver)\
    .withColumnRenamed('franchiseid','franchise_id_dim')\
    .withColumnRenamed('name','franchise_name')\
    .withColumnRenamed('city','franchise_city')\
    .withColumnRenamed('country','franchise_country')\
    .withColumnRenamed('size','franchise_size')

supplier_silver= f"{config['database']}sales_suppliers"
supplier_silver= spark.read.table(supplier_silver)\
    .withColumnRenamed('supplierid','supplier_id_dim')\
    .withColumnRenamed('name','supplier_name')\
    .withColumnRenamed('city','supplier_city')\
    .withColumnRenamed('country','supplier_country')\
    .withColumnRenamed('size','supplier_size')\
    .withColumnRenamed('continent','supplier_continent')

customer_silver=f"{config['database']}sales_customers"
customer_silver= spark.read.table(customer_silver)\
    .withColumnRenamed('customerid','customer_id_dim')

media_customer_reviews=f"{config['bronze_path']}media_customer_reviews"
# "/Volumes/inceptezcatalog/bakehouse/bake_house/bronze/media_customer_reviews"
media_customer_reviews=spark.read.format('delta').load(media_customer_reviews).withColumnRenamed('franchiseid','review_franchiseid').dropDuplicates()\
    .withColumn("clean_review",lower(col("review")))

# franchise_silver.show(3)
# supplier_silver.show(3)

In [0]:
'''
1.joining the transactional data with silver layer of franchise , suppliers and customers to get the final silver layer of transactions.
2.I am using left outer join to get all the records from transactional data and for non matching records in the master or dimentional table will be handled by DQC.
'''


transactions_silver= sales_transactions.join(franchise_silver, 
                                             sales_transactions.franchiseid == franchise_silver.franchise_id_dim, 'left')\
                                       .join(supplier_silver, 
                                             franchise_silver.supplierid == supplier_silver.supplier_id_dim, 'left')\
                                       .join(customer_silver, 
                                             sales_transactions.customerid == customer_silver.customer_id_dim, 'left')\
                                       .join(media_customer_reviews, 
                                             media_customer_reviews.review_franchiseid == franchise_silver.franchise_id_dim, 'left')\
                                       .withColumn("tran_month",trunc("datetime","month")).drop('ingestion_date')

# transactions_silver.count()

In [0]:
'''Data Quality Function used to validate the rules needs to set on certain columns
'''
from pyspark.sql import functions as F
def apply_dq_checks(df, dq_rules):
    error_cols = []

    for col_name, rules in dq_rules.items():

        for rule, message in rules.items():

            if rule == "not_null":
                error_col = F.when(F.col(col_name).isNull(), F.lit(message))
            
            elif rule == "positive":
                error_col = F.when(F.col(col_name) <= 0, F.lit(message))

            elif rule == "valid_date":
                error_col = F.when(F.col(col_name).isNull()|
                                   ((F.to_date(F.col(col_name)) < F.to_date(F.lit("2020-01-01")))|
                                    (F.to_date(F.col(col_name)) > F.current_date())), F.lit(message))

            else:
                continue

            error_cols.append(error_col)

    # Combine all error messages into an array
    df = df.withColumn(
        "dq_errors",
        F.array(*[c for c in error_cols])
    )

    # Filter out nulls inside the array
    df = df.withColumn(
        "dq_errors",
        F.expr("filter(dq_errors, x -> x is not null)")
    )

    # Add pass/fail flag
    df = df.withColumn(
        "dq_status",
        F.when(F.size("dq_errors") > 0, "FAIL").otherwise("PASS")
    )

    return df

In [0]:
'''Bad data from silver final transactional table will be stored into seperate Delta table, that will send for review or can be used to trigger mails
  here Both the data quality check and referential integrity check has placed in the final transactional data
  good data will be stored in the delta table for registering table in unity catalog,for data data/schema discovery,
  for processing it in the UI and schema enforcement, 
'''

silver_df = apply_dq_checks(transactions_silver, dq_rules)
bad_data_silver= f"{config['silver_path']}bad_data"
sales_tran_data= f"{config['silver_path']}sales_tran_data"
bad_df = silver_df.filter("dq_status = 'FAIL'").limit(1).count()
if bad_df >0:
    silver_df.filter("dq_status = 'FAIL'").write.format("delta")\
      .partitionBy("tran_month").mode("overwrite").saveAsTable(f"{config['database']}final_bad_tran_data")
# display(bad_df)
# bad_df.show(3)

'''Writing the final sales transactional data into the Silver layer'''
silver_df.write.format("delta").mode('overwrite').partitionBy("tran_month").saveAsTable(f"{config['database']}sales_tran_data")
print("Transaction load completed")